In [8]:
import pandas as pd
from pathlib import Path


In [14]:
file_paths = [
    "outputs/PPO_LR_attention_800/test_policy_results.csv",
    "outputs/BC_MLP_LR_ep800_epoch50/test_policy_results.csv",
    "outputs/BC_XGB_LR_ep800/test_policy_results.csv",
    "outputs/PPO_LR_gru_800/test_policy_results.csv",
    "outputs/PPO_XGB_attention_800/test_policy_results.csv"
]

dfs = []
for path in file_paths:
    df = pd.read_csv(path)[["policy", "cumulative_profit_mean", "liquidated_episode_count"]]
    df["source"] = Path(path).parent.name  # 어느 파일인지 표시
    dfs.append(df)

result = pd.concat(dfs, ignore_index=True)
result.to_csv("comparision_merged.csv", index=False)

- PPO - 100 updates
- BC MLP - 50 epochs
- BC XGB - n_estimators=200

### Source: PPO_LR_attention_800

| policy | cumulative_profit_me | liquidated |
| :--- | :---: | :---: |
| greedy_expected_profit | 8.46M | 0 |
| one_step_lp_closed_form | 8.46M | 0 |
| budget_aware_heuristic | 6.07M | 0 |
| fixed_conservative | 6.00M | 0 |
| fixed_aggressive | 4.88M | 0 |
| ppo_rl | 555.04K | 0 |
| reject_all | -3.58M | 0 |

### Source: BC_MLP_LR_ep800_epoch50

| policy | cumulative_profit_me | liquidated |
| :--- | :---: | :---: |
| greedy_expected_profit | 9.02M | 0 |
| one_step_lp_closed_form | 9.02M | 0 |
| budget_aware_heuristic | 6.44M | 0 |
| fixed_conservative | 6.36M | 0 |
| fixed_aggressive | 5.18M | 0 |
| reject_all | -3.58M | 0 |
| bc_model | 3.23M | 0 |

### Source: BC_XGB_LR_ep800

| policy | cumulative_profit_me | liquidated |
| :--- | :---: | :---: |
| greedy_expected_profit | 9.02M | 0 |
| one_step_lp_closed_form | 9.02M | 0 |
| budget_aware_heuristic | 6.44M | 0 |
| fixed_conservative | 6.36M | 0 |
| fixed_aggressive | 5.18M | 0 |
| reject_all | -3.58M | 0 |
| bc_model | -10.13M | 44 |

### Source: PPO_LR_gru_800

| policy | cumulative_profit_me | liquidated |
| :--- | :---: | :---: |
| greedy_expected_profit | 8.46M | 0 |
| one_step_lp_closed_form | 8.46M | 0 |
| budget_aware_heuristic | 6.07M | 0 |
| fixed_conservative | 6.00M | 0 |
| fixed_aggressive | 4.88M | 0 |
| ppo_rl | 977.13K | 0 |
| reject_all | -3.58M | 0 |

### Source: PPO_XGB_attention_800

| policy | cumulative_profit_me | liquidated |
| :--- | :---: | :---: |
| greedy_expected_profit | 8.46M | 0 |
| one_step_lp_closed_form | 8.46M | 0 |
| budget_aware_heuristic | 6.07M | 0 |
| fixed_conservative | 6.00M | 0 |
| fixed_aggressive | 4.88M | 0 |
| ppo_rl | 555.04K | 0 |
| reject_all | -3.58M | 0 |

### Best_case (No 파산 & 수익 max)

| Source | Policy | cumulative_profit_me | liquidated |
| ------ | ------ | -------------------- | ---------- |
| PPO_LR_attention_800 | greedy_expected_profit, one_step_lp_closed_form | 8.46M | 0 |
| BC_MLP_LR_ep800_epoch50 | greedy_expected_profit, one_step_lp_closed_form | 9.02M | 0 |
| BC_XGB_LR_ep800 | greedy_expected_profit, one_step_lp_closed_form | 9.02M  | 0 |
| PPO_LR_gru_800 | greedy_expected_profit, one_step_lp_closed_form | 8.46M | 0 |
| PPO_XGB_attention_800 | greedy_expected_profit, one_step_lp_closed_form | 8.46M | 0 |

### ML(BC) vs RL

| 실험 | PPO/BC 수익 | 파산 |
| :--- | :---: | :---: |
| PPO Attention (LR) | +555K | 0/100 |
| BC MLP | +3.23M | 0/100 |
| **BC XGB** | **-10.13M** | **44/100** |
| ***PPO GRU (LR)*** | ***+977K*** | ***0/100*** |
| PPO Attention (XGB) | +555K | 0/100 |

#### Sub1 (Attention vs GRU)

- GRU가 Attention보다 수익이 높아요 (+977K vs +555K).
- GRU가 30개 segment를 순차적으로 처리하면서 시계열 패턴을 더 잘 잡은 것으로 보임

#### Sub2 (LR vs XGBoost scorer)

- 수익이 완전히 동일해요 (+555K). upstream scorer 품질이 RL 성능에 거의 영향을 주지 않는다는 결론
- PPO가 segment calibration 파라미터를 직접 보면서 학습하기 때문에 risk score 품질에 덜 의존

#### Main (PPO vs BC)

- BC MLP
  - 파산 없이 +3.23M으로 PPO보다 높은 수익을 기록
  - budget_aware_heuristic을 모방했기 때문인데, heuristic 자체가 +6.44M이고 BC가 이를 어느 정도 따라간 결과
  - 다만 이게 "BC가 PPO보다 좋다"는 의미는 아님. BC는 heuristic이라는 정답지를 보고 모방한 지도학습이고, PPO는 정답 없이 스스로 학습
  - BC => ML => label 필요 => budget_aware_heuristic 모방
    ```python
    def collect_bc_data(env_factory, episodes, seed):
        ...
        while True:
            # heuristic이 결정한 allocation을 action label로 사용
            allocation = budget_aware_segment_policy(env)  # ← 여기
            states.append(obs.copy())
            actions.append(allocation.astype(np.float32))
    ```

- BC XGB
  - 44/100 파산, -10.13M으로 실패
  - MLP와 동일한 데이터로 학습했는데 성능이 크게 떨어진 이유는, XGBoost가 31차원 연속 출력을 독립적인 31개 회귀 문제로 분리해서 학습하기 때문
  - segment 간 상호의존성 (예: A에 많이 배분하면 B에 덜 배분해야 함)을 전혀 고려하지 못해서 allocation이 유동성 제약을 위반
